<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W4_J4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction du Taux de Désabonnement des Clients Bancaires (Customer Churn Prediction)

Ce notebook a pour objectif de construire un modèle d'apprentissage automatique pour prédire si des clients bancaires sont susceptibles de se désabonner (churn). Nous allons suivre les étapes classiques d'un projet de Machine Learning : exploration des données, prétraitement, entraînement d'un modèle de classification, et évaluation de ses performances.

## 1. Importation des Bibliothèques Nécessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_style('whitegrid')

## 2. Chargement des Données

Nous allons charger le jeu de données `Churn_Modelling.csv`. Si le fichier n'est pas déjà dans votre environnement Colab, veuillez l'uploader ou ajuster le chemin d'accès.

In [ ]:
# TODO: Assurez-vous que le fichier 'Churn_Modelling.csv' est bien présent ou ajustez le chemin d'accès.
try:
    df = pd.read_csv('Churn_Modelling.csv')
    print("Données chargées avec succès.")
except FileNotFoundError:
    print("Erreur: Le fichier 'Churn_Modelling.csv' est introuvable. Veuillez l'uploader ou vérifier le chemin.")
    # Créer un DataFrame vide pour éviter les erreurs subséquentes si le fichier n'est pas trouvé
    df = pd.DataFrame()

# Affichage des premières lignes pour un aperçu
display(df.head())

## 3. Exploration des Données (EDA)

Nous allons examiner la structure des données, les types de variables, les statistiques descriptives et la distribution de la variable cible.

In [ ]:
if not df.empty:
    print("Informations générales sur le DataFrame :")
    df.info()
    print("\nStatistiques descriptives des variables numériques :")
    display(df.describe())

    print("\nDistribution de la variable cible 'Exited' :")
    display(df['Exited'].value_counts(normalize=True))

    plt.figure(figsize=(6, 4))
    sns.countplot(x='Exited', data=df, palette='viridis')
    plt.title('Distribution du Taux de Désabonnement (Exited)')
    plt.xlabel('Désabonné (0 = Non, 1 = Oui)')
    plt.ylabel('Nombre de Clients')
    plt.show()
else:
    print("Le DataFrame est vide, impossible de réaliser l'EDA.")

### Analyse des Valeurs Manquantes

Il est crucial de vérifier et de gérer les valeurs manquantes avant d'entraîner un modèle.

In [ ]:
if not df.empty:
    print("Nombre de valeurs manquantes par colonne :")
    missing_values = df.isnull().sum()
    display(missing_values[missing_values > 0])

    if missing_values.sum() == 0:
        print("\nAucune valeur manquante détectée.")
    else:
        print("\nDes valeurs manquantes ont été détectées. Des stratégies d'imputation pourraient être nécessaires si ce n'est pas déjà géré.")
else:
    print("Le DataFrame est vide, impossible de vérifier les valeurs manquantes.")

## 4. Prétraitement des Données

Cette étape est cruciale pour préparer les données pour l'entraînement du modèle. Elle inclura :
- Suppression des colonnes non pertinentes (`CustomerId`, `Surname`, `RowNumber`).
- Encodage des variables catégorielles (`Geography`, `Gender`).
- Normalisation des variables numériques.

In [ ]:
if not df.empty:
    # Séparer les caractéristiques (X) de la variable cible (y)
    X = df.drop(['Exited', 'RowNumber', 'CustomerId', 'Surname'], axis=1)
    y = df['Exited']

    # Identifier les colonnes numériques et catégorielles
    numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X.select_dtypes(include='object').columns.tolist()

    print(f"Colonnes Numériques : {numerical_cols}")
    print(f"Colonnes Catégorielles : {categorical_cols}")

    # Créer un pipeline de prétraitement
    # Transformation pour les colonnes numériques (StandardScaler)
    numerical_transformer = StandardScaler()

    # Transformation pour les colonnes catégorielles (OneHotEncoder)
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')

    # Combiner les transformateurs en utilisant ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols)
        ])

    print("Prétraitement des données configuré.")
else:
    print("Le DataFrame est vide, impossible de procéder au prétraitement.")

## 5. Division des Données en Ensembles d'Entraînement et de Test

Nous allons diviser les données en un ensemble d'entraînement (80%) et un ensemble de test (20%) pour évaluer les performances du modèle sur des données non vues.

In [ ]:
if not df.empty:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Taille de l'ensemble d'entraînement X : {X_train.shape}")
    print(f"Taille de l'ensemble de test X : {X_test.shape}")
    print(f"Taille de l'ensemble d'entraînement y : {y_train.shape}")
    print(f"Taille de l'ensemble de test y : {y_test.shape}")
else:
    print("Le DataFrame est vide, impossible de diviser les données.")

## 6. Construction et Entraînement du Modèle

Nous allons construire un pipeline complet qui inclut le prétraitement et le modèle d'apprentissage automatique. Pour commencer, nous utiliserons une Régression Logistique, puis un RandomForestClassifier pour comparer.

### Modèle 1: Régression Logistique

In [ ]:
if not df.empty:
    # Créer un pipeline pour la Régression Logistique
    model_lr = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42, solver='liblinear')) # solver='liblinear' est bon pour les petits datasets
    ])

    # Entraîner le modèle
    model_lr.fit(X_train, y_train)
    print("Modèle de Régression Logistique entraîné.")
else:
    print("Le DataFrame est vide, impossible d'entraîner le modèle de Régression Logistique.")

### Modèle 2: Forêt Aléatoire (RandomForestClassifier)

In [ ]:
if not df.empty:
    # Créer un pipeline pour le RandomForestClassifier
    model_rf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, n_estimators=100)) # n_estimators: nombre d'arbres
    ])

    # Entraîner le modèle
    model_rf.fit(X_train, y_train)
    print("Modèle RandomForestClassifier entraîné.")
else:
    print("Le DataFrame est vide, impossible d'entraîner le modèle RandomForestClassifier.")

## 7. Évaluation du Modèle

Nous allons évaluer les performances de nos modèles sur l'ensemble de test en utilisant des métriques clés comme l'exactitude (accuracy), la précision (precision), le rappel (recall), le F1-score et l'AUC-ROC. Nous allons également visualiser la matrice de confusion.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] # Probabilités pour la classe positive (désabonnement)

    print(f"--- Évaluation du Modèle : {model_name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Désabonné', 'Désabonné'],
                yticklabels=['Non-Désabonné', 'Désabonné'])
    plt.xlabel('Prédictions')
    plt.ylabel('Vraies Valeurs')
    plt.title(f'Matrice de Confusion pour {model_name}')
    plt.show()

    return { 'accuracy': accuracy_score(y_test, y_pred),
             'precision': precision_score(y_test, y_pred),
             'recall': recall_score(y_test, y_pred),
             'f1_score': f1_score(y_test, y_pred),
             'roc_auc': roc_auc_score(y_test, y_pred_proba) }

### Évaluation de la Régression Logistique

In [ ]:
if 'model_lr' in locals(): # Vérifier si le modèle a été entraîné
    lr_metrics = evaluate_model(model_lr, X_test, y_test, "Régression Logistique")
else:
    print("Le modèle de Régression Logistique n'a pas été entraîné.")

### Évaluation du RandomForestClassifier

In [ ]:
if 'model_rf' in locals(): # Vérifier si le modèle a été entraîné
    rf_metrics = evaluate_model(model_rf, X_test, y_test, "RandomForestClassifier")
else:
    print("Le modèle RandomForestClassifier n'a pas été entraîné.")

## 8. Comparaison des Modèles et Interprétation

Nous allons comparer les performances des deux modèles et discuter de ce que nous pouvons apprendre de leurs résultats. L'interprétation du modèle est essentielle pour prendre des décisions commerciales éclairées.

In [ ]:
if 'lr_metrics' in locals() and 'rf_metrics' in locals():
    metrics_df = pd.DataFrame({
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'],
        'Logistic Regression': [lr_metrics['accuracy'], lr_metrics['precision'], lr_metrics['recall'], lr_metrics['f1_score'], lr_metrics['roc_auc']],
        'Random Forest': [rf_metrics['accuracy'], rf_metrics['precision'], rf_metrics['recall'], rf_metrics['f1_score'], rf_metrics['roc_auc']]
    })
    display(metrics_df.set_index('Metric'))

    print("\n--- Interprétation des Métriques ---")
    print("En général, le Recall est une métrique importante pour la prédiction de désabonnement, car nous voulons identifier autant de clients potentiels au désabonnement que possible (minimiser les faux négatifs) pour intervenir.\n")

    if rf_metrics['recall'] > lr_metrics['recall']:
        print("Le RandomForestClassifier semble avoir un meilleur rappel, ce qui est souvent préférable pour les problèmes de détection de désabonnement.")
    elif lr_metrics['recall'] > rf_metrics['recall']:
        print("La Régression Logistique semble avoir un meilleur rappel.")
    else:
        print("Les deux modèles ont un rappel similaire.")

    # Tentative d'interpréter les caractéristiques importantes pour RandomForest (plus facile que LR pour les coefficients)
    try:
        feature_names = model_rf.named_steps['preprocessor'].get_feature_names_out()
        importances = model_rf.named_steps['classifier'].feature_importances_

        forest_importances = pd.Series(importances, index=feature_names)

        fig, ax = plt.subplots(figsize=(10, 6))
        forest_importances.nlargest(10).plot.bar(ax=ax)
        ax.set_title("Top 10 Caractéristiques Importantes (RandomForest)")
        ax.set_ylabel("Importance Gini")
        plt.tight_layout()
        plt.show()

        print("\nLes caractéristiques importantes du RandomForest peuvent donner un aperçu des facteurs clés du désabonnement.")
        print("Par exemple, un score de crédit faible, un âge élevé, ou un faible solde bancaire sont souvent des indicateurs de désabonnement.")
    except Exception as e:
        print(f"Impossible d'obtenir les importances des caractéristiques pour RandomForest : {e}")

    print("\n--- Conclusions Commerciales ---")
    print("Sur la base de ces modèles, la banque peut cibler les clients à haut risque identifiés. Par exemple, si le solde du compte est un facteur important et que des clients avec des soldes faibles sont prédits de se désabonner, des offres de services ou de conseils financiers pourraient être proposées.")

else:
    print("Impossible de comparer les modèles car l'un ou les deux n'ont pas été entraînés ou évalués.")